In [ ]:
# Importar librerías para conectarse y trabajar con dataframes
from sqlalchemy import create_engine
import pandas as pd
import pyodbc
from urllib.parse import quote_plus
import os
from dotenv import load_dotenv

# Cargo las variables de conexión desde el archivo .env (override por si el kernel ya tenía valores cacheados)
load_dotenv(override=True)

In [ ]:
# Configurar la conexión a PostgreSQL de origen usando las variables del .env
usuario = os.getenv("PG_USUARIO")
password = os.getenv("PG_PASSWORD")
servidor = os.getenv("PG_SERVIDOR")
puerto = os.getenv("PG_PUERTO")
base_datos = os.getenv("PG_BASE_DATOS")

cadena = (
    f"postgresql+psycopg2://{usuario}:{password}@"
    f"{servidor}:{puerto}/{base_datos}"
)

engine_origen = create_engine(cadena)

try:
    with engine_origen.connect() as conexion:
        print("Conexión exitosa")
except Exception as error:
    print(type(error))
    print(error)

Conexión exitosa


In [3]:
# Obtener las subcategorías desde PostgreSQL
consulta_subcategoria = "SELECT * FROM tienda.subcategoria"
df_subcategoria = pd.read_sql_query(consulta_subcategoria, engine_origen)

In [ ]:
# Configurar la conexión al stage en SQL Server usando las variables del .env
servidor_sqlserver = os.getenv("SQLSERVER_SERVIDOR")
usuario_sqlserver = os.getenv("SQLSERVER_USUARIO")
password_sqlserver = os.getenv("SQLSERVER_PASSWORD")
base_datos_stage = os.getenv("SQLSERVER_BASE_DATOS")
esquema_stage = os.getenv("SQLSERVER_ESQUEMA")
driver_sqlserver = os.getenv("SQLSERVER_DRIVER")

cadena_sqlserver = (
    f"mssql+pyodbc://{usuario_sqlserver}:{quote_plus(password_sqlserver)}@"
    f"{servidor_sqlserver}/{base_datos_stage}?driver={quote_plus(driver_sqlserver)}"
)

engine_stage = create_engine(cadena_sqlserver)
try:
    with engine_stage.connect() as conexion:
        print("Conexión exitosa")
except Exception as error:
    print(type(error))
    print(error)

Conexión exitosa


In [5]:
# Agregar la fecha y hora de carga al dataframe
df_subcategoria["fecha_carga"] = pd.Timestamp.now()

In [6]:
# Revisar una muestra de las subcategorías antes de cargar
df_subcategoria.head()

,subcategoria_id,categoria_id,nombre,descripcion,activo,fecha_creacion,fecha_modificacion,fecha_carga
0,1,1,Figuras articuladas,Productos de figuras articuladas,True,2026-09-16 14:22:49.159577,2026-09-16 14:22:49.159577,2026-09-18 12:44:18.640963
1,2,1,Funko Pop,Productos de funko pop,True,2026-09-16 14:22:49.159577,2026-09-16 14:22:49.159577,2026-09-18 12:44:18.640963
2,3,1,Estatuas,Productos de estatuas,True,2026-09-16 14:22:49.159577,2026-09-16 14:22:49.159577,2026-09-18 12:44:18.640963
3,4,2,Manga,Productos de manga,True,2026-09-16 14:22:49.159577,2026-09-16 14:22:49.159577,2026-09-18 12:44:18.640963
4,5,2,Cómic americano,Productos de cómic americano,True,2026-09-16 14:22:49.159577,2026-09-16 14:22:49.159577,2026-09-18 12:44:18.640963


In [7]:
# Cargar las subcategorías en la tabla del stage
df_subcategoria.to_sql(
    name="subcategoria",
    con=engine_stage,
    schema=esquema_stage,
    if_exists="append",
    index=False
)

24